<a href="https://colab.research.google.com/github/Aayush974/learning-pytorch/blob/main/Cnns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Need for CNNs and the problem they solve

Giving an image to a dense MLP model changes its 2D structure into a 1d vector


![2d to 1d img.png](https://res.cloudinary.com/dkaeohu5t/image/upload/v1778920847/2d_to_1d_img_x6itbx.png)

this conversion of 2d image matrix to a 1d representation causes
1. Loss of spatial information
2. each pixel in the 1d array to be treated independent from others, there is no concept of neighbours.

now for a 2d structure like an image the spatial information is a valuable assest hence to preserve this spatial strucutre convolutional neural networks are used

# What is a convolution

In general terms, a convolution is a mathematical operation which combines two functions to create a third function.

The convolution is the bakcbone of CNNs hence the name
Even though it is heavily used in image processing , it is not limited to it. As stated above it is a mathematical operation.

Libraries like numpy actually offer convolution methods in them

In [ ]:
import numpy as np

arr1 = np.array([1,2,3])
arr2 = np.array([4,5,6])

np.convolve(arr1,arr2)

array([ 4, 13, 28, 27, 18])

given two function *f(x)* and *g(x)*, a convolution is defined as the sum of products of the points of collision btw *f(x)* and g'(x) where g'(x) is the mirror of *g(x)*,<br>
for example: if in the above example arr1 can be considered as values for some function *f(x)* and similarily arr2 represent *g(x)*, then its convolution is calucalted as shown in the image below

![conv.png](https://res.cloudinary.com/dkaeohu5t/image/upload/v1778920974/conv_g6jaj4.png)

## Need for Inversion of second function

the mirroring of the function is required by the definition, if the second function is not mirrored it is called **cross-correlation**
<br>
To get an idea as to why exactly it is required , think of it like this, Convolution is an operation which helps us see how two function correlate so for that to be performed the starting value of first function must be mapped over to the first value of second function, the second to second, third to third and so on
<br>
if the second function was not mirrored then the last value of second will be mapped with the first value , second-last with the seecond value and so on


**NOTE:**
A very common misconception is believing pyTorch's nn.Conv2d (explained below) performs convolution. It does NOT, rather it performs cross correlation

for 1d arrays convolution can also be though of as the sum of anti-diagonals elements of the matrix multiplication of the transpose of first array and the second array

In [ ]:
arr1 = np.array([arr1])
arr2 = np.array([arr2])
arr1.T@arr2

# sum of anti diagonals of this
# 4,8+5,12+10+6,15+12,18 = 4,13,28,27,18

array([[ 4,  5,  6],
       [ 8, 10, 12],
       [12, 15, 18]])

# Convolution in image processing

Now we are mostly concerned with how these convolutions are applied in neural networks to actually overcome the shortcoming of MLPs discussed above

A filter called the kernel is slid over the image vector , the elements of the filter and the overlapping elements of image matrix are elemenwise multiplied and then added to form a single element of what is called a "feature map"

![2d vector conv.png](https://res.cloudinary.com/dkaeohu5t/image/upload/v1778921038/2d_vector_conv_iqnlcj.png)
as shown in the image the input is a 5x5 image vector and the kernel is a 3x3 vector which is slid across the input vector to obtain element of the feature map

the full feature map is

![2d vector conv.png](https://res.cloudinary.com/dkaeohu5t/image/upload/v1778921037/2d_vector_conv_result_eiocxc.png)

the images taken above is from this lecture
[mit lecture on cnns by Alexander Amini](https://www.youtube.com/watch?v=iaSUYvmCekI&t=671s)

most of content of this notebook is just me writing down stuff i learned from this lecture


## significance of kernels/filters

each kernel acts as a filter to extract specific information from an image for example in the above picture, the applied filter is a filter used to detect if the input image is an image contain x , so the filter is "active" (1) on the diagonals and "inactive"(0) on the other parts.

this is just a simplied example of what kernels do and of course there are various types of kernels used to
1. detect edges
2. sharpen images
3. blur images etc



# CNNs for classification

a basic cnn architecture consists of these 3 layers

1. **Convolutional Layer**

    the convolutional layer applies convolutions to the input images to obtain various feature maps using different kernels. The number of kernels is specified by the users and it determines how many features will be extracted by the convolutional layer

2. **Non-linearity layer**

    the non linearity layers introduces non-linearity to the convoluted data. This layer often implements ReLU

3. **Pooling layer**

    the pooling layer is used to reduce the dimensionality of the data while maintaining the spatial information of the image.
    This is the main factor behind how a 62x62 image can be reduced to, say 10 classes at the output layer while preserving the features and info extracted by the previous layers
    
    There are various kinds of pooling methods
      - Max pooling
      - Min pooling
      - Average pooling etc

# Building a small CNN

## getting some dummy data

In [ ]:
import torch
from torch import nn
images = torch.randn(size=(32,3,64,64))
images.shape # batch,channel,height,width

torch.Size([32, 3, 64, 64])

## conv2d

nn.conv2D is used to create a convolutional layer in pytorch
it has the following parameters
- **in_channels**

    specifies the input channels
    - 1 for greyscale images
    - 3 for RGB images

- **out_channels**

    specifies the no. of kernels/feature maps which will be created , the output of the layer will have this many channels.
    It just defines the output channel/features number, the actual dimensions of those is dependent on other factors
  
- **kernel_size**

    the size of the kernel, common sizes include 3x3, 5x5 and 7x7 kernels

- **stride**

    how much the kernel will shift at each iteration of the convolution, typically kept to 1

- **padding**

    padding is used to bloat up the input vector with 0s at the edges, this is done to control the dimensions of the features map<br>
    for example if an input matrix is of shape (7,7) then a kernel of size (3,3) will produce a feature map of size (5,5) but by padding it by 1 pixels around the edges we can get a feature map of shape (7,7)

In [ ]:
conv = nn.Conv2d(
    in_channels=3,
    out_channels=4,
    kernel_size=3,
    stride=1,
    padding=1
)

y=conv(images)
print(y.shape)
print(y[0].shape)
print(y[0])

torch.Size([32, 4, 64, 64])
torch.Size([4, 64, 64])
tensor([[[-5.0332e-01, -5.1181e-01,  2.3496e-01,  ..., -1.0639e+00,
          -3.1432e-01, -9.9380e-02],
         [ 8.5945e-02, -3.9994e-02,  3.4570e-02,  ..., -3.5141e-01,
          -2.1098e-02, -2.0285e-01],
         [-7.0738e-01, -9.9168e-02,  1.3297e-01,  ...,  1.4652e+00,
           2.1005e-01,  5.8944e-01],
         ...,
         [ 9.8901e-01, -9.8098e-02, -1.2457e+00,  ...,  6.3300e-01,
          -5.9372e-01, -7.8452e-02],
         [-5.7876e-01, -4.5194e-01,  2.0184e-01,  ..., -9.9769e-01,
          -3.0746e-02,  3.4338e-01],
         [-1.7041e-01,  1.4692e-02, -4.7105e-01,  ...,  6.3412e-01,
           3.0654e-01, -1.7208e-01]],

        [[ 1.5655e-01, -1.1363e-02, -1.1029e-01,  ...,  6.4985e-01,
           3.7919e-01,  4.3397e-01],
         [ 1.2420e+00,  1.1569e+00,  4.4870e-03,  ...,  1.1859e+00,
           1.7141e+00, -6.0873e-01],
         [ 5.7794e-01,  7.7029e-01, -4.2559e-01,  ..., -4.5460e-01,
           1.2407e+00,  

if we change the out_channels,padding,stride and kernel_size the output shape changes
Following is a formula to predict the shape of the output:

---
For one dimension:

$$
\text{Output Size} =
\left\lfloor
\frac{W - K + 2P}{S}
\right\rfloor + 1
$$

Where:

- \(W\) = input size
- \(K\) = kernel size
- \(P\) = padding
- \(S\) = stride

The same formula is applied independently to:

- height
- width

---
Full Conv2D Output Shape

Input shape:

```python
(C_in, H_in, W_in)
```

Output shape:

```python
(C_out, H_out, W_out)
```

Where:

- `C_out = out_channels`

Height:

$$
H_{out} =
\left\lfloor
\frac{H_{in} - K_h + 2P_h}{S_h}
\right\rfloor + 1
$$

Width:

$$
W_{out} =
\left\lfloor
\frac{W_{in} - K_w + 2P_w}{S_w}
\right\rfloor + 1
$$

## Creating a model

In [ ]:
from torch import nn

class convModel_0(nn.Module):
  def __init__(self,input_units,hidden_units,output_units):
    super().__init__()
    # [32,3,64,64]
    self.layer1 = nn.Sequential(
        nn.Conv2d(
            in_channels=input_units,
            out_channels=hidden_units,
            kernel_size=3,
            padding=1,
            stride=1
        ),
        # [32,16,64,64]
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=3,
            padding=1,
            stride=1
        ),
        #[32,16,64,64]
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2) # this config of max pool halves the dimensions so h = h/2 same for width
        #[32,16,32,32]
    )

    self.layer2 = nn.Sequential(
        #[32,16,32,32]
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=3,
            padding=1,
            stride=1
        ),
        nn.ReLU(),
        nn.Conv2d(
            in_channels=hidden_units,
            out_channels=hidden_units,
            kernel_size=3,
            padding=1,
            stride=1
        ),
        #[32,16,32,32]
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2) # this config of max pool halves the dimensions so h = h/2 same for width
        #[32,16,16,16]
    )

    self.classifier = nn.Sequential(
        # [32,16,16,16]
        nn.Flatten(),
        #[32,4096]
        nn.Linear(in_features=hidden_units*16*16,out_features=output_units)
    )

  def forward(self,x):
    print(f"initial shape {x.shape}")
    x = self.layer1(x)
    print(f"shape after layer 1 {x.shape}")
    x = self.layer2(x)
    print(f"shape after layer 2 {x.shape}")
    x = self.classifier(x)
    print(f"shape after classifier {x.shape}")
    return x

model = convModel_0(
    input_units=3,
    hidden_units=16,
    output_units=10
)

y = model(images)

initial shape torch.Size([32, 3, 64, 64])
shape after layer 1 torch.Size([32, 16, 32, 32])
shape after layer 2 torch.Size([32, 16, 16, 16])
shape after classifier torch.Size([32, 10])


the above model implements 2 layer each of which contains
- convolutional layer
- non-linearlity layer
- max-pooling layer

after these layers is a calssifier layer which
- flatttens the image
- gives the desired output

# References

The main sources on which the contents of this notebook were made are

- [what is a convolution?](https://www.youtube.com/watch?v=KuXjwB4LzSA&t=890s)
- [formal definition of Convolution](https://en.wikipedia.org/wiki/Convolution)
- [Intro to CNNs](https://www.youtube.com/watch?v=iaSUYvmCekI&t=1710s)
